# PhysioNet 2019 Utility — Model Comparison

Compares **LogisticGLM vs XGBoost vs GRU** across training sizes and bootstrap sample sizes.

## Test Grid
| Parameter | Values |
|---|---|
| Models | LogisticGLM · XGBoost · GRU |
| Training patients | 50 · 100 · 200 · 300 |
| Bootstrap patients / sample | 25 · 50 · 100 |
| Bootstrap iterations | 5 per config (set `N_ITER` higher for production) |

**Note:** GRU takes longer per iteration (~1-2 min per iteration). Reduce `N_ITER` or `TRAIN_SIZES` if runtime is too long.

In [1]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys, logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

sys.path.insert(0, '.')
logging.basicConfig(level=logging.WARNING)

from config import Config, TrainingConfig, BootstrapConfig
from data_loader import (
    load_physionet_files,
    add_hours_until_sepsis,
    split_patients_by_status,
    get_rows_for_patients,
)
from bootstrap import BootstrapResampler
from models import LogisticGLM, XGBoostModel, GRUModel
from training import BootstrapEvaluator

print('Imports OK')

/Users/vrose/ClaudeContainer/venv311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


## 1  Load data

In [2]:
DATA_DIR = Path('../data/physionet_sepsis')

print('Loading PSV files …')
raw_df = load_physionet_files(DATA_DIR)
print(f'  {raw_df["patient_id"].nunique():,} patients · {len(raw_df):,} rows')

print('Computing hours_until_sepsis …')
df = add_hours_until_sepsis(raw_df, keep_post_onset=True)

n_septic = df['SepsisLabel'].sum()
print(f'  Septic rows: {n_septic:,}')
print(f'  Non-septic rows: {len(df) - n_septic:,}')
print(f'  Sepsis prevalence: {100*n_septic/len(df):.1f}%')

Loading PSV files …


  20,336 patients · 790,215 rows
Computing hours_until_sepsis …
  Septic rows: 17,136
  Non-septic rows: 773,079
  Sepsis prevalence: 2.2%


## 2  Experiment configuration

In [3]:
MODELS = [
    ('LogisticGLM', LogisticGLM),
    ('XGBoost', XGBoostModel),
    ('GRU', GRUModel),
]
TRAIN_SIZES = [50, 100, 200]  # Reduced for speed; add 300 if time permits
BOOTSTRAP_SIZES = [25, 50]
N_ITER = 5  # Reduced for faster iteration; use 10+ for production
RANDOM_STATE = 42

total_evals = len(MODELS) * len(TRAIN_SIZES) * len(BOOTSTRAP_SIZES) * N_ITER
print(f'Total configurations: {len(MODELS)} models × {len(TRAIN_SIZES)} train × {len(BOOTSTRAP_SIZES)} boot × {N_ITER} iter')
print(f'                    = {total_evals} total bootstrap evaluations')
print()
print('⏱️  Estimated time: ~10-15 minutes')
print('   (GRU is slower; adjust N_ITER or TRAIN_SIZES if needed)')

Total configurations: 3 models × 3 train × 2 boot × 5 iter
                    = 90 total bootstrap evaluations

⏱️  Estimated time: ~10-15 minutes
   (GRU is slower; adjust N_ITER or TRAIN_SIZES if needed)


## 3  Run bootstrap evaluation for all models

In [ ]:
records = []

pbar_models = tqdm(MODELS, desc='Model', position=0)

for model_name, model_class in pbar_models:
    pbar_models.set_postfix(current_model=model_name)

    pbar_train = tqdm(TRAIN_SIZES, desc=f'  Train size', position=1, leave=False)

    for train_size in pbar_train:
        pbar_train.set_postfix(n_train=train_size)

        # ── Patient split ─────────────────────────────────────────────────────
        train_pids, boot_pids = split_patients_by_status(
            df, n_train_patients=train_size,
            random_state=RANDOM_STATE, stratify_by_sepsis=True,
        )
        train_df = get_rows_for_patients(df, train_pids)

        pbar_boot = tqdm(BOOTSTRAP_SIZES, desc=f'    Boot size', position=2, leave=False)

        for boot_size in pbar_boot:
            # ── Bootstrap resampler ───────────────────────────────────────────
            resampler = BootstrapResampler(
                bootstrap_pool_patient_ids = boot_pids,
                full_df                    = df,
                n_iterations               = N_ITER,
                bootstrap_sample_size      = boot_size,
                random_state               = RANDOM_STATE,
            )

            # ── Fit model once on training set ──────────────────────────────
            evaluator = BootstrapEvaluator(
                model            = model_class(),
                train_df         = train_df,
                label_column     = 'SepsisLabel',
                patient_id_column = 'patient_id',
            )

            # ── Evaluate on each bootstrap sample ────────────────────────────
            for i in range(N_ITER):
                _, boot_df = resampler.generate_iteration(i)
                m = evaluator.evaluate_iteration(
                    boot_df, i,
                    compute_per_group = True,
                    group_column      = 'Gender',
                )

                row = dict(
                    model         = model_name,
                    train_size    = train_size,
                    boot_size     = boot_size,
                    iteration     = i,
                    n_samples     = m['n_samples'],
                    n_positive    = m['n_positive'],
                    prevalence    = m['prevalence'],
                    auroc         = m.get('auroc',    np.nan),
                    recall        = m.get('recall',   np.nan),
                    accuracy      = m.get('accuracy', np.nan),
                    f1            = m.get('f1',       np.nan),
                    utility       = m.get('utility',  np.nan),
                )

                # per-group utility
                if 'per_group' in m:
                    for g, gm in m['per_group'].items():
                        row[f'utility_g{int(g)}'] = gm.get('utility', np.nan)
                        row[f'auroc_g{int(g)}']   = gm.get('auroc',   np.nan)

                records.append(row)

results = pd.DataFrame(records)
print(f'✓  {len(results):,} rows collected')
results.head()

Model:   0%|          | 0/3 [00:00<?, ?it/s, current_model=LogisticGLM]
/Users/vrose/ClaudeContainer/venv311/lib/python3.11/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: [7]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/vrose/ClaudeContainer/venv311/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/vrose/ClaudeContainer/venv311/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/vrose/Cl

## 4  Aggregate by model and training size

In [ ]:
# Aggregate across bootstrap sizes and iterations
agg = results.groupby(['model', 'train_size'])[[
    'auroc', 'recall', 'accuracy', 'f1', 'utility'
]].agg(['mean', 'std']).round(4)

agg.columns = ['_'.join(c) for c in agg.columns]
agg = agg.reset_index()

print('Aggregated metrics by model and training size:')
print(agg[['model', 'train_size', 'utility_mean', 'utility_std', 
           'auroc_mean', 'recall_mean']].to_string(index=False))

## 5  Model comparison visualisations

In [ ]:
# 5a: Utility and AUROC vs training size (lines for each model)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for metric, ax, title in [
    ('utility', axes[0], 'Utility (PhysioNet 2019)'),
    ('auroc', axes[1], 'AUROC'),
]:
    for model_name in [m[0] for m in MODELS]:
        model_data = results[results['model'] == model_name]
        grouped = model_data.groupby('train_size')[metric].agg(['mean', 'std']).reset_index()
        ax.errorbar(grouped['train_size'], grouped['mean'], yerr=grouped['std'],
                   label=model_name, marker='o', capsize=5, lw=2)
    
    ax.set_xlabel('Training patients')
    ax.set_ylabel(title)
    ax.set_title(f'{title} vs Training Size')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_xticks(TRAIN_SIZES)

plt.tight_layout()
plt.savefig('model_comparison_metrics.png', dpi=150)
plt.show()
print('✓  model_comparison_metrics.png')

In [ ]:
# 5b: Utility distributions by model (violin/box plot)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=results, x='model', y='utility', ax=axes[0], palette='Set2')
axes[0].axhline(0, ls='--', color='red', lw=1.2, label='Inaction baseline')
axes[0].set_title('Utility Distribution by Model')
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Utility')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# AUROC by model
sns.boxplot(data=results, x='model', y='auroc', ax=axes[1], palette='Set3')
axes[1].set_title('AUROC Distribution by Model')
axes[1].set_xlabel('Model')
axes[1].set_ylabel('AUROC')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_distributions.png', dpi=150)
plt.show()
print('✓  model_distributions.png')

In [ ]:
# 5c: Heatmap: Model × Training Size for Utility
pivot_utility = results.groupby(['model', 'train_size'])['utility'].mean().unstack('train_size')

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot_utility, annot=True, fmt='.3f', cmap='RdYlGn', ax=ax,
           cbar_kws={'label': 'Mean Utility'})
ax.set_title('Mean Utility — Model × Training Size')
ax.set_xlabel('Training patients')
ax.set_ylabel('Model')
plt.tight_layout()
plt.savefig('utility_heatmap_model.png', dpi=150)
plt.show()
print('✓  utility_heatmap_model.png')

In [ ]:
# 5d: Model comparison table
summary = results.groupby('model')[['utility', 'auroc', 'recall', 'f1']].agg([
    ('mean', 'mean'),
    ('std', 'std'),
    ('min', 'min'),
    ('max', 'max'),
]).round(4)

print('\n' + '='*70)
print('MODEL SUMMARY STATISTICS')
print('='*70)
print(summary.to_string())

## 6  Statistical comparison

In [ ]:
# Best configuration for each model
print('\n' + '='*70)
print('BEST CONFIGURATIONS BY MODEL')
print('='*70)

for model_name in [m[0] for m in MODELS]:
    model_results = results[results['model'] == model_name]
    best_idx = model_results['utility'].idxmax()
    best = model_results.loc[best_idx]
    
    print(f'\n{model_name}:')
    print(f'  Train size: {int(best["train_size"])} patients')
    print(f'  Boot size: {int(best["boot_size"])} patients/sample')
    print(f'  Utility: {best["utility"]:.4f}')
    print(f'  AUROC: {best["auroc"]:.4f}')
    print(f'  Recall: {best["recall"]:.4f}')

In [ ]:
# Scaling analysis: how does each model improve with more training data?
min_train_size = TRAIN_SIZES[0]
max_train_size = TRAIN_SIZES[-1]

print('\n' + '='*70)
print('SCALING TRENDS (Utility improvement: min → max training size)')
print('='*70)

for model_name in [m[0] for m in MODELS]:
    model_results = results[results['model'] == model_name]
    by_train = model_results.groupby('train_size')['utility'].mean()
    
    min_u = by_train.iloc[0]
    max_u = by_train.iloc[-1]
    improvement = (max_u - min_u) / abs(min_u) * 100 if min_u != 0 else 0
    
    print(f'\n{model_name}:')
    print(f'  {min_train_size:3d}pt: {min_u:.4f}')
    print(f'  {max_train_size:3d}pt: {max_u:.4f}')
    print(f'  Improvement: {improvement:+.1f}%')

## 7  Export results

In [ ]:
results.to_csv('utility_model_comparison_results.csv', index=False)
agg.to_csv('utility_model_comparison_summary.csv', index=False)

print('✓  utility_model_comparison_results.csv')
print('✓  utility_model_comparison_summary.csv')

## 8  Key findings

In [ ]:
print('\n' + '='*70)
print('KEY FINDINGS')
print('='*70)

# Overall winner
best_overall = results.loc[results['utility'].idxmax()]
print(f'\nBest overall configuration:')
print(f'  Model: {best_overall["model"]}')
print(f'  Training: {int(best_overall["train_size"])} patients')
print(f'  Bootstrap: {int(best_overall["boot_size"])} patients/sample')
print(f'  Utility: {best_overall["utility"]:.4f}')
print(f'  AUROC: {best_overall["auroc"]:.4f}')
print(f'  Recall: {best_overall["recall"]:.4f}')

# Model rankings
print(f'\nModel rankings by mean utility:')
model_means = results.groupby('model')['utility'].mean().sort_values(ascending=False)
for i, (model, utility) in enumerate(model_means.items(), 1):
    print(f'  {i}. {model}: {utility:.4f}')

# Model rankings by AUROC
print(f'\nModel rankings by mean AUROC:')
model_auroc = results.groupby('model')['auroc'].mean().sort_values(ascending=False)
for i, (model, auroc) in enumerate(model_auroc.items(), 1):
    print(f'  {i}. {model}: {auroc:.4f}')

# Stability (low std = more stable)
print(f'\nModel stability (utility std — lower is more stable):')
model_std = results.groupby('model')['utility'].std().sort_values()
for i, (model, std) in enumerate(model_std.items(), 1):
    print(f'  {i}. {model}: {std:.4f}')